In [0]:
%pip install beautifulsoup4

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import re
import requests
from bs4 import BeautifulSoup
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, TimestampType, DoubleType
from urllib.parse import urljoin


In [0]:

# Create the volume if it doesn't exist
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.nyc_taxi_bronze
""")

BRONZE_PATH = "/Volumes/workspace/default/nyc_taxi_bronze"
TLC_PAGE_URL = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
MONTHS = ["2023-01", "2023-02", "2023-03", "2023-04", "2023-05"]

def get_available_files():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(TLC_PAGE_URL, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    parquet_links = []
    for link in soup.find_all('a', href=True):
        href = link['href'].strip()
        if 'yellow_tripdata' in href and href.endswith('.parquet'):
            full_url = urljoin(TLC_PAGE_URL, href)
            parquet_links.append(full_url)
    
    return parquet_links

def filter_files_by_months(files, target_months):
    filtered = []
    for file_url in files:
        for month in target_months:
            if f"yellow_tripdata_{month}" in file_url:
                filtered.append((month, file_url))
                break
    return filtered

def download_file(month, url):
    filename = f"yellow_tripdata_{month}.parquet"
    local_path = f"{BRONZE_PATH}/{filename}"
    
    print(f"Downloading {month} from {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print(f"Saved to {local_path}")
    return local_path

# Main execution
print("Fetching available files from TLC website")
available_files = get_available_files()
print(f"Found {len(available_files)} yellow taxi parquet files")

print(f"\nFiltering for target months: {MONTHS}")
target_files = filter_files_by_months(available_files, MONTHS)
print(f"Found {len(target_files)} matching files\n")

for month, url in target_files:
    download_file(month, url)

print(f"\n✓ Download complete {len(target_files)} files saved to {BRONZE_PATH}")

Fetching available files from TLC website
Found 208 yellow taxi parquet files

Filtering for target months: ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05']
Found 5 matching files

Saved to /Volumes/workspace/default/nyc_taxi_bronze/yellow_tripdata_2023-01.parquet
Saved to /Volumes/workspace/default/nyc_taxi_bronze/yellow_tripdata_2023-02.parquet
Saved to /Volumes/workspace/default/nyc_taxi_bronze/yellow_tripdata_2023-03.parquet
Saved to /Volumes/workspace/default/nyc_taxi_bronze/yellow_tripdata_2023-04.parquet
Saved to /Volumes/workspace/default/nyc_taxi_bronze/yellow_tripdata_2023-05.parquet

✓ Download complete 5 files saved to /Volumes/workspace/default/nyc_taxi_bronze


In [0]:
import logging
from typing import List
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, TimestampType, DoubleType


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class NYCTaxiSilverPipeline:
  
    def __init__(self, spark: SparkSession, bronze_path: str, target_table: str):
        self.spark = spark
        self.bronze_path = bronze_path
        self.target_table = target_table
        
        
        self.required_columns = [
            F.col("VendorID").cast(IntegerType()),
            F.col("passenger_count").cast(IntegerType()),
            F.col("total_amount").cast(DoubleType()),
            F.col("tpep_pickup_datetime").cast(TimestampType()),
            F.col("tpep_dropoff_datetime").cast(TimestampType())
        ]

    def read_bronze(self, months_pattern: str = "0[1-5]") -> DataFrame:
        """Leitura paralela e otimizada via wildcard path."""
        try:
            logger.info(f"Iniciando leitura dos arquivos no caminho: {self.bronze_path}")
           
            months = ["01", "02", "03", "04", "05"]
            df_list = []
            
            for month in months:
                df_temp = self.spark.read.parquet(f"{self.bronze_path}/yellow_tripdata_2023-{month}.parquet")
                df_list.append(df_temp)
            
            
            df_result = df_list[0]
            for df in df_list[1:]:
                df_result = df_result.unionAll(df)
            
            return df_result
        except Exception as e:
            logger.error(f"Erro crítico ao ler dados da camada Bronze: {str(e)}")
            raise

    def transform(self, df: DataFrame) -> DataFrame:
        """Aplica a seleção de colunas, regras de Data Quality e partição."""
        logger.info("Iniciando transformações e aplicação de regras de Data Quality.")
        return (
            df.select(self.required_columns)
            .filter(
                F.col("tpep_pickup_datetime").isNotNull() &
                (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime")) &
                (F.col("total_amount") >= 0.0) &
                (F.col("passenger_count") > 0) &
                F.col("tpep_pickup_datetime").between("2023-01-01 00:00:00", "2023-05-31 23:59:59")
            )
            .withColumn("pickup_year_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
        )

    def write_silver(self, df: DataFrame) -> None:
        """Grava os dados no formato Delta Lake com evolução de schema controlada."""
        try:
            logger.info(f"Gravando dados na tabela Delta: {self.target_table}")
            (
                df.write
                .format("delta")
                .mode("overwrite")
                .partitionBy("pickup_year_month")
                .option("overwriteSchema", "true")
                .saveAsTable(self.target_table)
            )
            logger.info("Camada Silver gravada com sucesso.")
        except Exception as e:
            logger.error(f"Falha na escrita da tabela Delta: {str(e)}")
            raise

    def run(self) -> None:
        """Orquestrador de execução do pipeline."""
        df_raw = self.read_bronze()
        df_transformed = self.transform(df_raw)
        self.write_silver(df_transformed)



if __name__ == "__main__":
    
    BRONZE_DIR = "/Volumes/workspace/default/nyc_taxi_bronze"
    TARGET_DELTA_TABLE = "workspace.default.nyc_taxi_silver"

    pipeline = NYCTaxiSilverPipeline(
        spark=spark, 
        bronze_path=BRONZE_DIR, 
        target_table=TARGET_DELTA_TABLE
    )
    pipeline.run()

INFO:__main__:Iniciando leitura dos arquivos no caminho: /Volumes/workspace/default/nyc_taxi_bronze
INFO:__main__:Iniciando transformações e aplicação de regras de Data Quality.
INFO:__main__:Gravando dados na tabela Delta: workspace.default.nyc_taxi_silver
INFO:__main__:Camada Silver gravada com sucesso.


In [0]:
df_monthly = (
    spark.table("workspace.default.nyc_taxi_silver")
    .groupBy("pickup_year_month")
    .agg(
        F.sum("total_amount").alias("monthly_revenue"),
        F.count("*").alias("trip_count"),
        F.avg("total_amount").alias("avg_ticket"),
    )
)

(
    df_monthly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.nyc_taxi_gold_monthly")
)

print("✓ Gold table created successfully: workspace.default.nyc_taxi_gold_monthly")

✓ Gold table created successfully: workspace.default.nyc_taxi_gold_monthly


Cálculo considerando o ticket médio por corrida agrupado por mês:

In [0]:
%sql
SELECT 
  pickup_year_month,
  AVG(total_amount) as ticket_medio_corrida,
  COUNT(*) as total_corridas
FROM workspace.default.nyc_taxi_silver
GROUP BY pickup_year_month
ORDER BY pickup_year_month;

pickup_year_month,ticket_medio_corrida,total_corridas
2023-01,27.457851106515488,2918665
2023-02,27.365490001788956,2765061
2023-03,28.284956619576114,3227960
2023-04,28.77978551206451,3110852
2023-05,29.44905391433838,3320355


Cálculo considerando a média dos valores totais:

In [0]:
%sql
WITH faturamento_mensal AS (
  SELECT 
    pickup_year_month,
    SUM(total_amount) as faturamento_total
  FROM workspace.default.nyc_taxi_silver
  GROUP BY pickup_year_month
)

SELECT 
  AVG(faturamento_total) as media_faturamento_mensal_frota
FROM faturamento_mensal;

media_faturamento_mensal_frota
8.688423868977627E7


In [0]:
%sql
WITH passageiros_por_dia_hora AS (

  SELECT 
    DATE(tpep_pickup_datetime) as dia,
    HOUR(tpep_pickup_datetime) as hora,
    SUM(passenger_count) as total_passageiros_na_hora
  FROM workspace.default.nyc_taxi_silver
  WHERE pickup_year_month = '2023-05'
  GROUP BY DATE(tpep_pickup_datetime), HOUR(tpep_pickup_datetime)
)

SELECT 
  hora,
  AVG(total_passageiros_na_hora) as media_passageiros_por_hora
FROM passageiros_por_dia_hora
GROUP BY hora
ORDER BY hora;

hora,media_passageiros_por_hora
0,4078.6774193548385
1,2668.1612903225805
2,1737.6129032258063
3,1128.1935483870968
4,713.0
5,753.8709677419355
6,1849.1612903225807
7,3793.6129032258063
8,5241.741935483871
9,5960.4838709677415
